# 경사하강법의 함정 — 학습률·국소 최솟값·신경망 학습이 같은 원리인 이유 (2부)

**Zero to 머신러닝 딥러닝 Master — Medium 시리즈 #23 · 2부** · 원문: 책 01-10 「편미분·그래디언트·경사하강법 (2부)」

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/otoloism/Zero_to_ML_Master/blob/main/medium-series/23-2-learning-rate-local-minima/23_2_learning_rate_local_minima.ipynb)

학습률 0.01 / 0.1 / 0.25 / 0.5 / 0.9 로 30걸음씩 걸어 수렴·정확 도착·진동·발산을 비교하고, 웅덩이가 두 개인 f(x) = x⁴ − 4x² + 0.5x 에서 시작점에 따라 전역/지역 최솟값으로 갈리는 것을 확인한 뒤, 연습문제의 일반화 그래디언트와 수치 그래디언트를 구현합니다.

- 📝 Medium 글: **TODO: Medium 게시 후 URL 입력** (`https://medium.com/p/...`)
- 📘 책 원문: [01-10 편미분·그래디언트·경사하강법 (2부)](https://wikidocs.net/439754)
- 🏠 [mldict.net](https://mldict.net) · ✍️ [기술 블로그](https://wikidocs.net/blog/@mldict/) · 📚 [책 전체 목차](https://wikidocs.net/book/21464)

## 01-10-H 8단계 — PyTorch 학습 루프의 세 줄

optimizer.zero_grad() → loss.backward() → optimizer.step(). 모델·손실이 정의되어 있어야 하는 설명용 코드입니다.

```python
optimizer.zero_grad()   # 지난 걸음의 기울기 기록을 지운다 (grad는 누적되므로!)
loss.backward()       # ⬅️ 역전파: 100만 개 편미분을 한 번에 계산 → ∇L 완성
optimizer.step()        # 👣 w ← w - η∇L : 5단계에서 직접 만든 그 한 줄!
```

## 1. 01-10-F 6단계 — 학습률 5종 비교 실험

같은 시작점 (4, 3)에서 학습률만 바꿔 30걸음 후 위치와 손실을 비교합니다.

> ⚠️ 이 코드는 1부(5단계)의 `compute_loss`·`compute_gradient` 를 사용하므로, 그 정의를 앞에 **추가**했습니다(표시된 부분).

In [1]:
# [추가 코드 — 책 본문에는 없음] 1부(01-10-E 5단계)에서 정의한 지형 함수와 그래디언트
import numpy as np

def compute_loss(position):
    """지형의 높이(손실값)를 계산합니다."""
    x, y = position
    return x ** 2 + 2 * y ** 2

def compute_gradient(position):
    """∇f = (∂f/∂x, ∂f/∂y) = (2x, 4y)"""
    x, y = position
    return np.array([2 * x, 4 * y])

In [2]:
for lr in [0.01, 0.1, 0.25, 0.5, 0.9]:
    position = np.array([4.0, 3.0])
    for step in range(30):
        position = position - lr * compute_gradient(position)
    print(f"lr={lr}: x={position[0]:.6g}, y={position[1]:.6g}, "
          f"f={compute_loss(position):.6g}")

lr=0.01: x=2.18194, y=0.881573, f=6.31519
lr=0.1: x=0.00495176, y=6.63222e-07, f=2.45199e-05
lr=0.25: x=3.72529e-09, y=0, f=1.38778e-17
lr=0.5: x=0, y=3, f=18
lr=0.9: x=0.00495176, y=8.4396e+12, f=1.42454e+26


## 2. 01-10-G 7단계 — 시작점에 따라 갈리는 운명 (지역 최솟값)

같은 알고리즘·같은 학습률인데 시작점만 다르게 해서 전역 최솟값과 지역 최솟값에 각각 도착하는 것을 확인합니다.

In [3]:
def f(x):
    """웅덩이가 두 개인 지형 (왼쪽이 진짜 바닥)"""
    return x**4 - 4*x**2 + 0.5*x

def df(x):
    """도함수: 4x³ - 8x + 0.5"""
    return 4*x**3 - 8*x + 0.5

for start in [-2.0, -0.5, 0.5, 2.0]:      # 시작점만 다르게!
    x = start
    for _ in range(200):
        x = x - 0.01 * df(x)                 # 완전히 동일한 알고리즘
    print(f"start={start:>5} -> x={x:.4f}, f={f(x):.4f}")

start= -2.0 -> x=-1.4445, f=-4.7148
start= -0.5 -> x=-1.4445, f=-4.7148
start=  0.5 -> x=1.3819, f=-3.3009
start=  2.0 -> x=1.3819, f=-3.3009


## 3. 연습문제 — f(x, y) = a·x² + b·y² 의 그래디언트

계수 a, b 를 받는 일반화된 compute_gradient 입니다.

In [4]:
def compute_gradient(position, a, b):
    """f(x,y) = a·x² + b·y² 의 그래디언트"""
    x, y = position
    return np.array([2 * a * x, 2 * b * y])

## 4. 연습문제 — 수치 그래디언트

각 방향으로 ±h 만큼 밀어 보며 기울기를 재는 '실측' 방식입니다.

> ⚠️ 연습문제 해설 코드는 함수 정의만 있어, 실습 1(점 (2, 1))로 공식과 수치 그래디언트를 비교하는 코드를 **추가**했습니다(표시된 부분).

In [5]:
def numerical_gradient(f, position, h=1e-5):
    """각 방향으로 살짝 밀어보며 기울기를 재는 '실측' 방식"""
    grad = np.zeros_like(position)
    for i in range(len(position)):
        p_plus, p_minus = position.copy(), position.copy()
        p_plus[i] += h
        p_minus[i] -= h
        grad[i] = (f(p_plus) - f(p_minus)) / (2 * h)
    return grad

In [6]:
# [추가 코드 — 책 본문에는 없음] 실습 1 — f(x, y) = 3x² + y² 의 점 (2, 1)에서 공식 vs 수치 그래디언트
f_xy = lambda p: 3 * p[0] ** 2 + p[1] ** 2
pos = np.array([2.0, 1.0])
print("공식 그래디언트 (6x, 2y):", compute_gradient(pos, 3, 1))
print("수치 그래디언트        :", numerical_gradient(f_xy, pos))

공식 그래디언트 (6x, 2y): [12.  2.]
수치 그래디언트        : [12.  2.]


## 🔗 함께 보기

- 📘 **책 (WikiDocs)** — [Zero to 머신러닝 딥러닝 Master](https://wikidocs.net/book/21464)
- 🏠 **홈페이지** — [mldict.net](https://mldict.net)
- ✍️ **기술 블로그** — [wikidocs.net/blog/@mldict](https://wikidocs.net/blog/@mldict/)
- 💻 **전체 코드 저장소** — [github.com/otoloism/Zero_to_ML_Master](https://github.com/otoloism/Zero_to_ML_Master)

👉 더 자세한 설명과 그림은 [책 원문](https://wikidocs.net/439754) 에서 확인하세요.